# Select G7 + China Datasets from Stanford AI Index 2026

This notebook scans all CSV files under `data/number/PUBLIC DATA_ 2026 AI INDEX REPORT/` and selects datasets that contain at least 4 of the G7 + China countries.

**G7 countries**: United States, United Kingdom, France, Germany, Italy, Japan, Canada
**Plus**: China


## 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd
import re

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').is_dir():
            return p
    raise FileNotFoundError('Could not locate repo root containing `data/`')

repo_root = find_repo_root(Path.cwd())
data_root = repo_root / 'data' / 'number' / 'PUBLIC DATA_ 2026 AI INDEX REPORT'

print(f'Repo root: {repo_root}')
print(f'Data root: {data_root}')
print(f'Data root exists: {data_root.exists()}')


## 2. Define Target Countries

We use full and short names because the Stanford dataset uses both forms (e.g. "United States" vs "United States of America").


In [ ]:
TARGET_COUNTRIES = {
    'United States': ['United States', 'United States of America'],
    'China': ['China'],
    'United Kingdom': ['United Kingdom', 'United Kingdom of Great Britain and Northern Ireland'],
    'France': ['France'],
    'Germany': ['Germany'],
    'Italy': ['Italy'],
    'Japan': ['Japan'],
    'Canada': ['Canada'],
}

# Minimum number of target countries a file must contain to be included
MIN_COUNTRY_COUNT = 4

print(f'Target countries: {list(TARGET_COUNTRIES.keys())}')
print(f'Minimum match: {MIN_COUNTRY_COUNT} countries')


## 3. Helper Functions

In [ ]:
def read_csv_safely(path: Path) -> pd.DataFrame:
    """Read CSV with encoding fallback."""
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding='latin1')


def find_countries_in_file(csv_path: Path) -> list:
    """Return the list of target countries that appear in this CSV."""
    try:
        # Read raw text to capture country names anywhere (column headers, cells, etc.)
        text = csv_path.read_text(encoding='utf-8', errors='replace')
    except Exception:
        return []
    
    found = []
    for canonical, aliases in TARGET_COUNTRIES.items():
        for alias in aliases:
            if alias in text:
                found.append(canonical)
                break
    return found


def get_file_metadata(csv_path: Path) -> dict:
    """Return metadata about a CSV file."""
    try:
        df = read_csv_safely(csv_path)
        return {
            'rows': len(df),
            'columns': len(df.columns),
            'column_names': ', '.join(map(str, df.columns)),
        }
    except Exception as e:
        return {'rows': None, 'columns': None, 'column_names': f'ERROR: {e}'}


print('Helper functions loaded')


## 4. Scan All CSV Files

In [ ]:
records = []

for csv_path in sorted(data_root.glob('*/Data/*.csv')):
    countries_found = find_countries_in_file(csv_path)
    
    if len(countries_found) >= MIN_COUNTRY_COUNT:
        meta = get_file_metadata(csv_path)
        records.append({
            'chapter': csv_path.parents[1].name,
            'figure': csv_path.stem,
            'country_count': len(countries_found),
            'countries_found': '; '.join(countries_found),
            'rows': meta['rows'],
            'columns': meta['columns'],
            'column_names': meta['column_names'],
            'relative_path': csv_path.relative_to(repo_root).as_posix(),
        })

result_df = pd.DataFrame(records).sort_values(
    ['chapter', 'country_count'],
    ascending=[True, False]
).reset_index(drop=True)

print(f'Found {len(result_df)} datasets containing >= {MIN_COUNTRY_COUNT} target countries')
result_df.head(10)


## 5. Summary by Chapter

In [ ]:
chapter_summary = result_df.groupby('chapter').agg(
    dataset_count=('figure', 'count'),
    avg_countries=('country_count', 'mean'),
    max_countries=('country_count', 'max'),
).round(2)

chapter_summary


## 6. Datasets Containing All 8 Countries

In [ ]:
full_coverage = result_df[result_df['country_count'] == 8]
print(f'{len(full_coverage)} datasets contain all 8 target countries')
full_coverage[['chapter', 'figure', 'rows', 'columns', 'column_names']]


## 7. Save the Output

In [ ]:
output_csv = repo_root / 'data' / 'number'/ 'g7_china_datasets.csv'
result_df.to_csv(output_csv, index=False)
print(f'Saved to: {output_csv}')
print(f'Total rows: {len(result_df)}')


## 8. Generate Markdown Summary

In [ ]:
def generate_markdown_summary(df: pd.DataFrame) -> str:
    lines = [
        '# G7 + China AI Datasets',
        '',
        'Datasets from the Stanford AI Index 2026 public data that contain at least '
        f'{MIN_COUNTRY_COUNT} of the G7 + China countries.',
        '',
        '**G7 countries**: United States, United Kingdom, France, Germany, Italy, Japan, Canada  ',
        '**Plus**: China',
        '',
        '---',
        '',
    ]
    
    for chapter in sorted(df['chapter'].unique()):
        sub = df[df['chapter'] == chapter].sort_values('country_count', ascending=False)
        lines.append(f'## {chapter} ({len(sub)} datasets)')
        lines.append('')
        lines.append('| Figure | Country Count | Columns |')
        lines.append('|--------|---------------|---------|')
        for _, row in sub.iterrows():
            cols = str(row['column_names']).replace('|', '\\|')
            lines.append(f"| {row['figure']} | {row['country_count']} | {cols} |")
        lines.append('')
    
    return '\n'.join(lines)


markdown_text = generate_markdown_summary(result_df)
output_md = repo_root / 'data' / 'number' / 'g7_china_datasets_summary.md'
output_md.write_text(markdown_text, encoding='utf-8')
print(f'Saved to: {output_md}')


## 9. Preview Full Result

In [ ]:
result_df
